In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

API_KEY = "yiwqJ5O0oZJrhsad9X4kxoUeCItqIllb"
TICKERS = ["SPY", "QQQ"]

BASE_URL = "https://api.polygon.io"

def fetch_1m_2years(ticker, start_date="2024-04-01", end_date="2026-04-01"):
    all_data = []

    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    current = start

    print(f"\nCapturing {ticker}...")

    while current <= end:
        day = current.strftime("%Y-%m-%d")

        url = (
            f"{BASE_URL}/v2/aggs/ticker/{ticker}/range/1/minute/"
            f"{day}/{day}?adjusted=true&sort=asc&limit=50000&apiKey={API_KEY}"
        )

        try:
            res = requests.get(url, timeout=30)
            data = res.json()
        except Exception as e:
            print(f"{day} request failed: {e}")
            current += timedelta(days=1)
            continue

        results = data.get("results", [])

        if results:
            all_data.extend(results)
            print(f"{day}: {len(results)} rows")
        else:
            print(f"{day}: no data")

        current += timedelta(days=1)

        time.sleep(15)

    df = pd.DataFrame(all_data)

    df["timestamp"] = pd.to_datetime(df["t"], unit="ms", utc=True)\
        .dt.tz_convert("America/New_York")

    df = df.set_index("timestamp")

    df = df[["o", "h", "l", "c", "v"]].rename(columns={
        "o": "Open",
        "h": "High",
        "l": "Low",
        "c": "Close",
        "v": "Volume"
    })

    df = df.sort_index()
    df = df[~df.index.duplicated()]

    return df


for ticker in TICKERS:
    df = fetch_1m_2years(ticker)
    df.to_csv(f"{ticker}_1m_2year.csv")
    print(f"{ticker} saved: {len(df)} rows")

print("\nDone")


开始抓取 SPY...
2024-04-01: 无数据
2024-04-02: 无数据
2024-04-03: 790 条
2024-04-04: 809 条
2024-04-05: 759 条
2024-04-06: 无数据
2024-04-07: 无数据
2024-04-08: 715 条
2024-04-09: 759 条
2024-04-10: 763 条
2024-04-11: 805 条
2024-04-12: 820 条
2024-04-13: 无数据
2024-04-14: 无数据
2024-04-15: 853 条
2024-04-16: 836 条
2024-04-17: 837 条
2024-04-18: 826 条
2024-04-19: 857 条
2024-04-20: 无数据
2024-04-21: 无数据
2024-04-22: 845 条
2024-04-23: 842 条
2024-04-24: 878 条
2024-04-25: 881 条
2024-04-26: 765 条
2024-04-27: 无数据
2024-04-28: 无数据
2024-04-29: 766 条
2024-04-30: 801 条
2024-05-01: 818 条
2024-05-02: 828 条
2024-05-03: 792 条
2024-05-04: 无数据
2024-05-05: 无数据
2024-05-06: 778 条
2024-05-07: 749 条
2024-05-08: 744 条
2024-05-09: 726 条
2024-05-10: 774 条
2024-05-11: 无数据
2024-05-12: 无数据
2024-05-13: 729 条
2024-05-14: 704 条
2024-05-15: 767 条
2024-05-16: 783 条
2024-05-17: 725 条
2024-05-18: 无数据
2024-05-19: 无数据
2024-05-20: 702 条
2024-05-21: 691 条
2024-05-22: 761 条
2024-05-23: 758 条
2024-05-24: 696 条
2024-05-25: 无数据
2024-05-26: 无数据
2024-05-27: 无数据

### 1min data

In [4]:
import pandas as pd

qqq = pd.read_csv("QQQ_1m_2year.csv")
spy = pd.read_csv("SPY_1m_2year.csv")

qqq["timestamp"] = pd.to_datetime(qqq["timestamp"], utc=True)
spy["timestamp"] = pd.to_datetime(spy["timestamp"], utc=True)

qqq = qqq.sort_values("timestamp").drop_duplicates(subset="timestamp")
spy = spy.sort_values("timestamp").drop_duplicates(subset="timestamp")

qqq = qqq.rename(columns={
    "Open": "QQQ_Open",
    "High": "QQQ_High",
    "Low": "QQQ_Low",
    "Close": "QQQ_Close",
    "Volume": "QQQ_Volume"
})

spy = spy.rename(columns={
    "Open": "SPY_Open",
    "High": "SPY_High",
    "Low": "SPY_Low",
    "Close": "SPY_Close",
    "Volume": "SPY_Volume"
})

df = pd.merge(qqq, spy, on="timestamp", how="inner")

df = df.set_index("timestamp").sort_index()

df = df.tz_convert("America/New_York")


df_ny = df.between_time("09:30", "16:00")


df_ny = df_ny.reset_index()

df_ny["timestamp"] = df_ny["timestamp"].dt.tz_localize(None)

df_ny.to_csv("SPYQQQ_1m.csv", index=False)

print("QQQ rows:", len(qqq))
print("SPY rows:", len(spy))
print("Overlap rows:", len(df))
print("NY session rows:", len(df_ny))
print(df_ny.tail())

QQQ rows: 435201
SPY rows: 415818
Overlap rows: 393282
NY session rows: 194423
                 timestamp  QQQ_Open  QQQ_High  QQQ_Low  QQQ_Close  \
194418 2026-04-01 15:56:00    584.03    584.42   583.96     584.16   
194419 2026-04-01 15:57:00    584.15    584.29   584.03     584.25   
194420 2026-04-01 15:58:00    584.25    584.48   584.25     584.33   
194421 2026-04-01 15:59:00    584.36    584.54   583.70     584.26   
194422 2026-04-01 16:00:00    584.32    584.60   584.26     584.33   

          QQQ_Volume  SPY_Open  SPY_High  SPY_Low  SPY_Close    SPY_Volume  
194418  2.809285e+05   655.015   655.350   654.94    655.070  6.582393e+05  
194419  3.598700e+05   655.070   655.255   654.96    655.235  6.985748e+05  
194420  6.653396e+05   655.230   655.455   655.22    655.360  9.454180e+05  
194421  2.304518e+06   655.360   655.510   654.72    655.170  1.365341e+06  
194422  2.308034e+05   655.230   655.600   655.19    655.310  2.104960e+05  


### 5m data

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("SPYQQQ_1m.csv", parse_dates=["timestamp"])
df = df.set_index("timestamp")


# resample to 5-minute

df_5m = df.resample("5min").agg({
    "SPY_Open": "first",
    "SPY_High": "max",
    "SPY_Low": "min",
    "SPY_Close": "last",
    "SPY_Volume": "sum",
    "QQQ_Open": "first",
    "QQQ_High": "max",
    "QQQ_Low": "min",
    "QQQ_Close": "last",
    "QQQ_Volume": "sum",
}).dropna().reset_index()

print("5m rows:", len(df_5m))
print(df_5m.tail())

df_5m.to_csv("SPYQQQ_5m.csv")

5m rows: 39287
                timestamp  SPY_Open  SPY_High   SPY_Low  SPY_Close  \
39282 2026-04-01 15:40:00    656.22    656.75  656.1150    656.155   
39283 2026-04-01 15:45:00    656.15    656.34  655.8699    655.980   
39284 2026-04-01 15:50:00    656.00    656.12  655.0300    655.240   
39285 2026-04-01 15:55:00    655.25    655.51  654.7200    655.170   
39286 2026-04-01 16:00:00    655.23    655.60  655.1900    655.310   

         SPY_Volume  QQQ_Open  QQQ_High  QQQ_Low  QQQ_Close    QQQ_Volume  
39282  1.064012e+06    584.91    585.51   584.86    584.930  6.820164e+05  
39283  2.093741e+06    584.92    585.25   584.80    585.045  1.043720e+06  
39284  2.705432e+06    585.07    585.22   584.04    584.370  1.818263e+06  
39285  4.246421e+06    584.38    584.54   583.70    584.260  3.887854e+06  
39286  2.104960e+05    584.32    584.60   584.26    584.330  2.308034e+05  


### Real-Time SMT

real-time SMT detects structure using only past and current data

In [40]:
import pandas as pd
import numpy as np

# =========================
# load data
# =========================
df = pd.read_csv("SPYQQQ_1m.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# =========================
# STRICT swing detection (NO FUTURE)
# =========================
def detect_swings_realtime(df, lookback=3):
    out = df.copy()

    # swing high: 当前 high > 过去 N 根的最高
    out["SPY_swing_high"] = (
        out["SPY_High"] > out["SPY_High"].rolling(lookback).max().shift(1)
    )

    out["QQQ_swing_high"] = (
        out["QQQ_High"] > out["QQQ_High"].rolling(lookback).max().shift(1)
    )

    # swing low: 当前 low < 过去 N 根的最低
    out["SPY_swing_low"] = (
        out["SPY_Low"] < out["SPY_Low"].rolling(lookback).min().shift(1)
    )

    out["QQQ_swing_low"] = (
        out["QQQ_Low"] < out["QQQ_Low"].rolling(lookback).min().shift(1)
    )

    return out


# =========================
# SMT logic (REAL-TIME)
# =========================
def detect_smt_realtime(df):
    out = df.copy()

    # last confirmed swings（只用过去）
    out["SPY_last_high"] = out["SPY_High"].where(out["SPY_swing_high"]).ffill()
    out["QQQ_last_high"] = out["QQQ_High"].where(out["QQQ_swing_high"]).ffill()

    out["SPY_last_low"] = out["SPY_Low"].where(out["SPY_swing_low"]).ffill()
    out["QQQ_last_low"] = out["QQQ_Low"].where(out["QQQ_swing_low"]).ffill()

    # previous swings
    out["SPY_prev_high"] = out["SPY_last_high"].shift(1)
    out["QQQ_prev_high"] = out["QQQ_last_high"].shift(1)

    out["SPY_prev_low"] = out["SPY_last_low"].shift(1)
    out["QQQ_prev_low"] = out["QQQ_last_low"].shift(1)

    # -------------------------
    # SMT conditions
    # -------------------------

    # bearish
    out["bearish_smt"] = (
        (out["SPY_last_high"] > out["SPY_prev_high"]) &
        (out["QQQ_last_high"] <= out["QQQ_prev_high"])
    ) | (
        (out["QQQ_last_high"] > out["QQQ_prev_high"]) &
        (out["SPY_last_high"] <= out["SPY_prev_high"])
    )

    # bullish
    out["bullish_smt"] = (
        (out["SPY_last_low"] < out["SPY_prev_low"]) &
        (out["QQQ_last_low"] >= out["QQQ_prev_low"])
    ) | (
        (out["QQQ_last_low"] < out["QQQ_prev_low"]) &
        (out["SPY_last_low"] >= out["SPY_prev_low"])
    )

    # 去重（避免连续触发）
    out["bearish_smt"] = out["bearish_smt"] & (~out["bearish_smt"].shift(1).fillna(False))
    out["bullish_smt"] = out["bullish_smt"] & (~out["bullish_smt"].shift(1).fillna(False))

    return out


# =========================
# backtest (clean summary)
# =========================
def backtest_summary(df, holding=5):
    out = df.copy()

    out["future_close"] = out["SPY_Close"].shift(-holding)

    bullish_ret = (out["future_close"] - out["SPY_Close"]) / out["SPY_Close"]
    bearish_ret = (out["SPY_Close"] - out["future_close"]) / out["SPY_Close"]

    returns = pd.concat([
        bullish_ret[out["bullish_smt"]],
        bearish_ret[out["bearish_smt"]]
    ])

    returns = returns.dropna()

    return {
        "total_signals": len(returns),
        "win_rate": (returns > 0).mean(),
        "avg_return": returns.mean(),
        "median_return": returns.median()
    }


df = detect_swings_realtime(df, lookback=3)
df = detect_smt_realtime(df)

summary = backtest_summary(df, holding=5)
print(summary)

df_5m = detect_swings_realtime(df_5m, lookback=3)
df_5m = detect_smt_realtime(df_5m)

summary_5m_rt = backtest_summary(df_5m, holding=5)
print(summary_5m_rt)

{'total_signals': 41171, 'win_rate': np.float64(0.5020767044764519), 'avg_return': np.float64(3.7844348394406574e-06), 'median_return': 7.273679463486787e-06}
{'total_signals': 8372, 'win_rate': np.float64(0.48100812231247014), 'avg_return': np.float64(1.7848190375815595e-05), 'median_return': -4.5301108601373234e-05}


### Centered SMT

uses a symmetric rolling window to identify swing highs and lows. A point is defined as a swing high only if it is higher than both past and future prices within a fixed window.

In [39]:
import pandas as pd
import numpy as np

# =========================
# LOAD 1m DATA
# =========================
df_1m = pd.read_csv("SPYQQQ_1m.csv", parse_dates=["timestamp"])
df_1m = df_1m.set_index("timestamp").sort_index()


# =========================
# RESAMPLE → 5m
# =========================
df_5m = df_1m.resample("5min").agg({
    "SPY_Open": "first",
    "SPY_High": "max",
    "SPY_Low": "min",
    "SPY_Close": "last",
    "SPY_Volume": "sum",
    "QQQ_Open": "first",
    "QQQ_High": "max",
    "QQQ_Low": "min",
    "QQQ_Close": "last",
    "QQQ_Volume": "sum",
}).dropna()


# =========================
# CENTERED SMT
# =========================
def detect_smt_centered(df, window=5):
    out = df.copy()

    # centered swing (⚠️ uses future)
    out["SH_SPY"] = out["SPY_High"] == out["SPY_High"].rolling(window*2+1, center=True).max()
    out["SH_QQQ"] = out["QQQ_High"] == out["QQQ_High"].rolling(window*2+1, center=True).max()

    out["SL_SPY"] = out["SPY_Low"] == out["SPY_Low"].rolling(window*2+1, center=True).min()
    out["SL_QQQ"] = out["QQQ_Low"] == out["QQQ_Low"].rolling(window*2+1, center=True).min()

    # swing prices
    out["SPY_swing_high"] = np.where(out["SH_SPY"], out["SPY_High"], np.nan)
    out["QQQ_swing_high"] = np.where(out["SH_QQQ"], out["QQQ_High"], np.nan)

    out["SPY_swing_low"] = np.where(out["SL_SPY"], out["SPY_Low"], np.nan)
    out["QQQ_swing_low"] = np.where(out["SL_QQQ"], out["QQQ_Low"], np.nan)

    # previous swings
    out["SPY_prev_high"] = pd.Series(out["SPY_swing_high"], index=out.index).ffill().shift(1)
    out["QQQ_prev_high"] = pd.Series(out["QQQ_swing_high"], index=out.index).ffill().shift(1)

    out["SPY_prev_low"] = pd.Series(out["SPY_swing_low"], index=out.index).ffill().shift(1)
    out["QQQ_prev_low"] = pd.Series(out["QQQ_swing_low"], index=out.index).ffill().shift(1)

    # SMT
    out["bearish_smt"] = (
        (out["SH_SPY"] & (out["SPY_High"] > out["SPY_prev_high"]) & (out["QQQ_High"] <= out["QQQ_prev_high"])) |
        (out["SH_QQQ"] & (out["QQQ_High"] > out["QQQ_prev_high"]) & (out["SPY_High"] <= out["SPY_prev_high"]))
    )

    out["bullish_smt"] = (
        (out["SL_SPY"] & (out["SPY_Low"] < out["SPY_prev_low"]) & (out["QQQ_Low"] >= out["QQQ_prev_low"])) |
        (out["SL_QQQ"] & (out["QQQ_Low"] < out["QQQ_prev_low"]) & (out["SPY_Low"] >= out["SPY_prev_low"]))
    )

    return out


# =========================
# BACKTEST (SUMMARY ONLY)
# =========================
def backtest_summary(df, holding=5):
    out = df.copy()

    out["future_close"] = out["SPY_Close"].shift(-holding)

    bullish_ret = (out["future_close"] - out["SPY_Close"]) / out["SPY_Close"]
    bearish_ret = (out["SPY_Close"] - out["future_close"]) / out["SPY_Close"]

    returns = pd.concat([
        bullish_ret[out["bullish_smt"]],
        bearish_ret[out["bearish_smt"]]
    ]).dropna()

    return {
        "total_signals": len(returns),
        "win_rate": (returns > 0).mean(),
        "avg_return": returns.mean(),
        "median_return": returns.median()
    }


# =========================
# RUN 1m CENTERED
# =========================
df_1m_c = detect_smt_centered(df_1m, window=5)
summary_1m_c = backtest_summary(df_1m_c, holding=5)

print("1m_centered:", summary_1m_c)


# =========================
# RUN 5m CENTERED
# =========================
df_5m_c = detect_smt_centered(df_5m, window=5)
summary_5m_c = backtest_summary(df_5m_c, holding=5)

print("5m_centered:", summary_5m_c)

1m_centered: {'total_signals': 6978, 'win_rate': np.float64(0.8445113212955001), 'avg_return': np.float64(0.0005338751397551719), 'median_return': 0.00034692238930442805}
5m_centered: {'total_signals': 1437, 'win_rate': np.float64(0.8455114822546973), 'avg_return': np.float64(0.001360182745004797), 'median_return': 0.0008688241086277995}


#### split data

Since this is time-series data, we cannot use random train-test splits because that would introduce look-ahead bias. Instead, we use a chronological split where the training set strictly precedes the test set. This ensures that the evaluation mimics a real trading scenario, where only past information is available when making decisions.

In [8]:
import pandas as pd

df = pd.read_csv("SPYQQQ_1m.csv", parse_dates=["timestamp"])

df = df.sort_values("timestamp")

split_date = "2025-07-01"

train_1m = df[df["timestamp"] < split_date]
test_1m  = df[df["timestamp"] >= split_date]

train_1m.to_csv("train_1m.csv", index=False)
test_1m.to_csv("test_1m.csv", index=False)

In [9]:
train_1m = train_1m.set_index("timestamp")
test_1m  = test_1m.set_index("timestamp")

def resample_5m(df):
    return df.resample("5min").agg({
        "SPY_Open": "first",
        "SPY_High": "max",
        "SPY_Low": "min",
        "SPY_Close": "last",
        "SPY_Volume": "sum",
        "QQQ_Open": "first",
        "QQQ_High": "max",
        "QQQ_Low": "min",
        "QQQ_Close": "last",
        "QQQ_Volume": "sum",
    }).dropna()

train_5m = resample_5m(train_1m)
test_5m  = resample_5m(test_1m)

train_5m.to_csv("train_5m.csv")
test_5m.to_csv("test_5m.csv")